# Sesi 3 – Data Cleaning: Missing, Outlier & Ekstraksi

| Keterangan | Detail |
|---|---|
| **Nama Lengkap** | [Farraz Dirgham H] |
| **NIM** | [240401020170] |
| **Kelas** | [IF405] |
| **Mata Kuliah** | Data Science |
| **Topik** | Data Cleaning – Missing Values, Outlier, Ekstraksi Data |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
np.random.seed(42)
print('Library siap!')

## 1. Membuat Dataset "Kotor" untuk Simulasi

In [ ]:
n = 150
data = {
    'id'        : range(1, n + 1),
    'nama'      : [f'Pelanggan_{i}' for i in range(1, n + 1)],
    'usia'      : np.random.randint(18, 65, n).astype(float),
    'gaji'      : np.random.normal(6_000_000, 1_800_000, n).round(),
    'kota'      : np.random.choice(['Jakarta', 'Bandung', 'Surabaya', 'Medan'], n),
    'tgl_daftar': pd.date_range('2023-01-01', periods=n, freq='3D').astype(str)
}
df = pd.DataFrame(data)

# Suntikkan missing values
for col in ['usia', 'gaji']:
    idx = np.random.choice(df.index, size=12, replace=False)
    df.loc[idx, col] = np.nan

# Suntikkan outlier ekstrem pada gaji
outlier_idx = np.random.choice(df.dropna(subset=['gaji']).index, size=5, replace=False)
df.loc[outlier_idx, 'gaji'] = df.loc[outlier_idx, 'gaji'] * 8

# Suntikkan duplikat
df = pd.concat([df, df.sample(6, random_state=1)], ignore_index=True)

print(f'Shape dataset: {df.shape}')
df.head(8)

## 2. Deteksi Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct})
print('=== Laporan Missing Values ===')
print(missing_report[missing_report['Jumlah Missing'] > 0])

In [ ]:
# Visualisasi pola missing values
fig, ax = plt.subplots(figsize=(8, 4))
missing_only = missing[missing > 0]
ax.bar(missing_only.index, missing_only.values, color='#F44336', edgecolor='white')
ax.set_title('Jumlah Missing Values per Kolom', fontweight='bold')
ax.set_ylabel('Jumlah Missing')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Penanganan Missing Values

In [ ]:
df_clean = df.copy()

# Strategi 1: Imputasi usia dengan median (lebih tahan outlier dibanding mean)
median_usia = df_clean['usia'].median()
df_clean['usia'].fillna(median_usia, inplace=True)
print(f'Usia diimputasi dengan median: {median_usia}')

# Strategi 2: Imputasi gaji dengan median per kota (lebih kontekstual)
df_clean['gaji'] = df_clean.groupby('kota')['gaji'].transform(
    lambda x: x.fillna(x.median())
)
print('Gaji diimputasi dengan median per kota.')

print(f'\nTotal missing setelah penanganan: {df_clean.isnull().sum().sum()}')

## 4. Deteksi & Penanganan Duplikat

In [ ]:
print(f'Jumlah baris duplikat: {df_clean.duplicated().sum()}')

sebelum = len(df_clean)
df_clean.drop_duplicates(inplace=True)
sesudah = len(df_clean)
print(f'Duplikat dihapus: {sebelum - sesudah} baris')
print(f'Shape setelah hapus duplikat: {df_clean.shape}')

## 5. Deteksi Outlier dengan Metode IQR

In [ ]:
Q1 = df_clean['gaji'].quantile(0.25)
Q3 = df_clean['gaji'].quantile(0.75)
IQR = Q3 - Q1
batas_bawah = Q1 - 1.5 * IQR
batas_atas  = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean['gaji'] < batas_bawah) | (df_clean['gaji'] > batas_atas)]
print(f'Q1 = {Q1:,.0f} | Q3 = {Q3:,.0f} | IQR = {IQR:,.0f}')
print(f'Batas bawah = {batas_bawah:,.0f} | Batas atas = {batas_atas:,.0f}')
print(f'\nJumlah outlier ditemukan: {len(outliers)}')
print(outliers[['id', 'gaji']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].boxplot(df_clean['gaji'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Boxplot Gaji (Sebelum Penanganan Outlier)', fontweight='bold')
axes[0].set_ylabel('Gaji (Rp)')

# Penanganan outlier: capping (winsorizing) ke batas IQR
df_clean['gaji_capped'] = df_clean['gaji'].clip(lower=batas_bawah, upper=batas_atas)

axes[1].boxplot(df_clean['gaji_capped'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightgreen'))
axes[1].set_title('Boxplot Gaji (Setelah Capping IQR)', fontweight='bold')
axes[1].set_ylabel('Gaji (Rp)')

plt.tight_layout()
plt.savefig('outlier_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Ekstraksi Data dari Kolom Teks/Tanggal

In [ ]:
# Ekstraksi angka dari kolom teks 'nama' (Pelanggan_123 -> 123)
df_clean['id_dari_nama'] = df_clean['nama'].str.extract(r'(\d+)').astype(int)
print('Ekstraksi angka dari kolom nama:')
print(df_clean[['nama', 'id_dari_nama']].head())

In [ ]:
# Ekstraksi komponen tanggal
df_clean['tgl_daftar'] = pd.to_datetime(df_clean['tgl_daftar'])
df_clean['tahun_daftar']  = df_clean['tgl_daftar'].dt.year
df_clean['bulan_daftar']  = df_clean['tgl_daftar'].dt.month
df_clean['hari_nama']     = df_clean['tgl_daftar'].dt.day_name()

print('Ekstraksi komponen tanggal:')
print(df_clean[['tgl_daftar', 'tahun_daftar', 'bulan_daftar', 'hari_nama']].head())

## 7. Standarisasi Format Teks

In [ ]:
# Simulasi data kota dengan format tidak konsisten
kota_kotor = pd.Series([' jakarta ', 'BANDUNG', 'Surabaya', 'medan ', 'Jakarta'])
print('Sebelum standarisasi:', list(kota_kotor))

kota_bersih = kota_kotor.str.strip().str.title()
print('Setelah standarisasi:', list(kota_bersih))

## 8. Validasi Akhir Dataset Bersih

In [ ]:
print('=== RINGKASAN DATASET SETELAH CLEANING ===')
print(f'Shape akhir       : {df_clean.shape}')
print(f'Missing values    : {df_clean.isnull().sum().sum()}')
print(f'Duplikat          : {df_clean.duplicated().sum()}')
print(f'\nStatistik gaji (setelah capping):')
print(df_clean['gaji_capped'].describe().round(0))

## 9. Kesimpulan

Pada Sesi 3 ini, saya mempelajari teknik-teknik *data cleaning*:

| Masalah Data | Teknik Penanganan |
|---|---|
| Missing Values | Imputasi median, imputasi median per grup |
| Duplikat | Deteksi dan penghapusan dengan `drop_duplicates()` |
| Outlier | Deteksi IQR, penanganan dengan *capping/winsorizing* |
| Ekstraksi Teks | Regex `str.extract()` untuk mengambil pola tertentu |
| Ekstraksi Tanggal | `dt.year`, `dt.month`, `dt.day_name()` |
| Standarisasi Teks | `str.strip()`, `str.title()` untuk konsistensi format |

> **Insight:** Data mentah hampir selalu "kotor". Kemampuan mendeteksi dan menangani missing values, outlier, dan inkonsistensi format adalah keterampilan fundamental sebelum data dapat dianalisis lebih jauh.